In [1]:
# 1. Install Unsloth and training dependencies
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

import os
# SAFEGUARD 1: Force single GPU to prevent DDP crash on Kaggle dual-T4
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel, get_chat_template
from unsloth.chat_templates import standardize_sharegpt
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from kaggle_secrets import UserSecretsClient
import torch


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-snfslizx/unsloth_d55652e4298e4da6a39df4969c430725
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-snfslizx/unsloth_d55652e4298e4da6a39df4969c430725
  Resolved https://github.com/unslothai/unsloth.git to commit b89f170dcae41d9a755c72aa77c8f25684eccf02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.1 MB/s eta 0:00

In [3]:
# 2. Authenticate with Hugging Face via Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token


In [4]:
# 3. Load Phase 2 Adapter Checkpoint
max_seq_length = 4096
dtype = None # Auto-detects float16 for T4 GPU
load_in_4bit = True 

phase2_model_id = "SiddardhaShayini/Granite-Claude-Code-Phase-2"

print(f"Loading Phase 2 checkpoint from {phase2_model_id}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = phase2_model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

# Put adapter into active training mode and re-arm Unsloth gradient checkpointing
model = FastLanguageModel.for_training(model, use_gradient_checkpointing = "unsloth")

# Verify trainable LoRA parameters
model.print_trainable_parameters()


Loading Phase 2 checkpoint from SiddardhaShayini/Granite-Claude-Code-Phase-2...
==((====))==  Unsloth 2026.10.3: Fast Granite patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Unsloth: ibm-granite/granite-3.1-2b-instruct had a bad pad_token (<|end_of_text|>). Using pad_token = <fim_pad>.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


trainable params: 28,180,480 || all params: 2,561,712,128 || trainable%: 1.1001


In [5]:
# 4. Load Phase 3 Datasets
data_dir = "/kaggle/input/datasets/siddardhashayini3/granite-claude-data/"

print("Loading 6 Phase 3 Agentic Datasets...")
ds_repo     = load_dataset("json", data_files=data_dir + "04_repository.jsonl", split="train")
ds_plan     = load_dataset("json", data_files=data_dir + "05_planning.jsonl", split="train")
ds_terminal = load_dataset("json", data_files=data_dir + "08_terminal.jsonl", split="train")
ds_git      = load_dataset("json", data_files=data_dir + "09_git.jsonl", split="train")
ds_tooluse  = load_dataset("json", data_files=data_dir + "10_tooluse.jsonl", split="train")
ds_swe      = load_dataset("json", data_files=data_dir + "11_swe_agent.jsonl", split="train")

# Concatenate, shuffle, and clean
phase3_dataset = concatenate_datasets([
    ds_repo,
    ds_plan,
    ds_terminal,
    ds_git,
    ds_tooluse,
    ds_swe
]).shuffle(seed=42)

phase3_dataset = phase3_dataset.filter(lambda x: len(x.get("conversations", [])) > 0)
print(f"Total Phase 3 Examples: {len(phase3_dataset)}")


Loading 6 Phase 3 Agentic Datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Total Phase 3 Examples: 7500


In [7]:
# 5. Format conversations with ChatML and map agentic roles
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

# Explicitly map agentic / tool-use roles into standard ChatML roles
phase3_dataset = standardize_sharegpt(
    phase3_dataset,
    aliases_for_system    = ["system"],
    aliases_for_user      = ["user", "human", "input", "observation", "tool", "environment"],
    aliases_for_assistant = ["gpt", "assistant", "output", "agent"],
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

formatted_dataset = phase3_dataset.map(formatting_prompts_func, batched = True)
print("✅ Dataset successfully standardized and formatted!")


[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|im_end|>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_u9j2mpkf/tokenizer_config.json.


Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/7500 [00:00<?, ? examples/s]

Map:   0%|          | 0/7500 [00:00<?, ? examples/s]

✅ Dataset successfully standardized and formatted!


In [8]:
# 6. Configure SFTTrainer for Phase 3
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Prevents OOM spikes on long agentic contexts
        gradient_accumulation_steps = 8, # Effective batch size = 8
        warmup_steps = 40,
        num_train_epochs = 1,            # 1 epoch covers all ~9K examples (~1,125 steps)
        learning_rate = 1e-4,            # Balanced LR for agentic tool use & XML formatting
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs_phase3",
        save_strategy = "no",            # Saves Kaggle disk space
        report_to = "none",
    ),
)

# Clear VRAM cache before starting
torch.cuda.empty_cache()


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/7500 [00:00<?, ? examples/s]

In [9]:
# 7. Start Training
print("Starting Phase 3 Training (Agentic Tool Use)...")
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 49155}.


Starting Phase 3 Training (Agentic Tool Use)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,500 | Num Epochs = 1 | Total steps = 938
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 28,180,480 of 2,561,712,128 (1.10% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.554266
20,1.273703
30,1.011608
40,0.665932
50,0.509536
60,0.498795
70,0.451537
80,0.453170
90,0.456289
100,0.402068


In [10]:
# 8. Push Final Model Adapter
your_hf_username = "SiddardhaShayini"
repo_name = f"{your_hf_username}/Granite-Claude-Code-Final-Agent"

print(f"Pushing Final Agent LoRA adapter to {repo_name}...")
model.push_to_hub(repo_name, token = hf_token)
tokenizer.push_to_hub(repo_name, token = hf_token)

print("✅ Final Agent Model Saved to Hugging Face!")


Pushing Final Agent LoRA adapter to SiddardhaShayini/Granite-Claude-Code-Final-Agent...


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpa3311xbz/tokenizer_config.json.


Saved model to https://huggingface.co/SiddardhaShayini/Granite-Claude-Code-Final-Agent
✅ Final Agent Model Saved to Hugging Face!


In [11]:
# Optional: Push merged full weights (float16) to a separate HF repository
merged_repo = f"{your_hf_username}/Granite-Claude-Code-Final-Agent-Merged"
print(f"Exporting and pushing merged 16-bit model to {merged_repo}...")

model.push_to_hub_merged(
    merged_repo,
    tokenizer,
    save_method = "merged_16bit",
    token = hf_token,
)
print("✅ Standalone merged model uploaded!")


Exporting and pushing merged 16-bit model to SiddardhaShayini/Granite-Claude-Code-Final-Agent-Merged...


Unsloth: Restored added_tokens_decoder metadata in SiddardhaShayini/Granite-Claude-Code-Final-Agent-Merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...



Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:12<01:12, 72.94s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:16<00:00, 38.49s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/SiddardhaShayini/Granite-Claude-Code-Final-Agent-Merged`
✅ Standalone merged model uploaded!
